In [1]:
import pandas as pd
import numpy as np

from sklearn.neighbors import BallTree
from pathlib import Path
from pyproj import Transformer
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

EARTH_RADIUS_M = 6_371_000
MAX_DIST_M = 75  # 30m prod default; 50m for houses (more scattered DPE)

In [2]:
# 1) DVF first (lighter RAM) — run estimate ~2m
dvf_dir = Path("../../ml/csv_data/dvf_plus")

DVF_USECOLS = [
    "datemut",
    "anneemut",
    "valeurfonc",
    "sbati",
    "sterr",
    "codtypbien",
    "idnatmut",
    "l_codinsee",
    "nblocdep",
    "nbmai1pp",
    "nbmai2pp",
    "nbmai3pp",
    "nbmai4pp",
    "nbmai5pp",
    "geompar_x",
    "geompar_y",
]

dfs_dvf = [
    pd.read_csv(path, sep="|", usecols=DVF_USECOLS, low_memory=False)
    for path in sorted(dvf_dir.glob("*.csv"))
]

df_dvf = pd.concat(dfs_dvf, ignore_index=True)
del dfs_dvf

df_dvf.shape

(16565022, 16)

In [3]:
# lon / lat from Lambert-93
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

df_dvf["lon"], df_dvf["lat"] = transformer.transform(
    df_dvf["geompar_x"].to_numpy(),
    df_dvf["geompar_y"].to_numpy(),
)

In [4]:
# Filter houses only (111 = house)
df_dvf_filtered = df_dvf[
    (df_dvf["idnatmut"] == 1)  # 1 = sale
    & (df_dvf["codtypbien"] == 111)
    & (df_dvf["anneemut"] >= 2023)
    & (df_dvf["valeurfonc"] > 10_000)
    & (df_dvf["sbati"] > 10)
    # France metropolitan
    & (df_dvf["lon"] >= -5.5)
    & (df_dvf["lon"] <= 10)
    & (df_dvf["lat"] >= 41)
    & (df_dvf["lat"] <= 51.5)
    & ((df_dvf["valeurfonc"] / df_dvf["sbati"]).between(500, 15_000))
    & (df_dvf["sterr"] > 0)
    & (df_dvf["sterr"] <= 50_000)
].copy()

del df_dvf
df_dvf_filtered.drop(columns=["geompar_x", "geompar_y"], inplace=True, errors="ignore")

df_dvf_filtered.shape

(1174403, 16)

In [5]:
df_dvf_filtered["home_rooms"] = (
    1 * df_dvf_filtered["nbmai1pp"]
  + 2 * df_dvf_filtered["nbmai2pp"]
  + 3 * df_dvf_filtered["nbmai3pp"]
  + 4 * df_dvf_filtered["nbmai4pp"]
  + 5 * df_dvf_filtered["nbmai5pp"]
)

df_dvf_filtered.drop(
    columns=["nbmai1pp", "nbmai2pp", "nbmai3pp", "nbmai4pp", "nbmai5pp"],
    inplace=True,
)

# sterr > 0 for houses → log is safe
df_dvf_filtered["log_sterr"] = np.log(df_dvf_filtered["sterr"])

In [6]:
# 2) DPE after DVF filter — run estimate ~1-2m
DPE_PATH = Path("../../ml/csv_data/dpe/dpe-france.csv")
DPE_LABEL_TO_SCORE = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

dpe_cols = [
    "numero_dpe",
    "date_etablissement_dpe",
    "classe_consommation_energie",
    "annee_construction",
    "latitude",
    "longitude",
    "tr001_modele_dpe_type_libelle",
    "tr002_type_batiment_description",
]

df_dpe = pd.read_csv(DPE_PATH, usecols=dpe_cols, low_memory=False)

df_dpe = df_dpe[
    df_dpe["tr001_modele_dpe_type_libelle"].isin(
        ["Vente", "Location", "Neuf", "Copropriété"]
    )
]

df_dpe["date_etablissement_dpe"] = pd.to_datetime(
    df_dpe["date_etablissement_dpe"], errors="coerce"
)
today = pd.Timestamp.today().normalize()
current_year = today.year

df_dpe = df_dpe[df_dpe["date_etablissement_dpe"] <= today]
df_dpe = df_dpe[df_dpe["annee_construction"].between(1850, current_year)]

df_dpe["dpe_median"] = (
    df_dpe["classe_consommation_energie"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map(DPE_LABEL_TO_SCORE)
)

df_dpe = df_dpe.rename(columns={"latitude": "lat", "longitude": "lon"})
df_dpe = df_dpe.dropna(subset=["dpe_median", "annee_construction", "lat", "lon"])
df_dpe = df_dpe.drop_duplicates(subset=["numero_dpe"], keep="first")

# DPE: standalone house + Logement (excluding collective buildings)
building_type = df_dpe["tr002_type_batiment_description"].astype(str)
df_dpe_maison = df_dpe[
    building_type.str.contains("Maison", case=False, na=False)
    | (
        (building_type == "Logement")
        & ~building_type.str.contains("collectif", case=False, na=False)
    )
]

dpe_bat = (
    df_dpe_maison.groupby(["lat", "lon"], as_index=False)
    .agg(
        dpe_median=("dpe_median", "median"),
        annee_construction=("annee_construction", "first"),
    )
)
dpe_bat["dpe_median"] = dpe_bat["dpe_median"].round().astype(int)

del df_dpe, df_dpe_maison

print("DPE maison buildings:", len(dpe_bat))

DPE maison buildings: 2798231


In [7]:
def merge_dpe_on_dvf(
    dvf_df: pd.DataFrame,
    dpe_bat: pd.DataFrame,
    max_dist_m: float = MAX_DIST_M,
) -> pd.DataFrame:
    result = dvf_df.copy()
    result["dpe_median"] = np.nan
    result["annee_construction"] = np.nan

    valid = result.dropna(subset=["lat", "lon"])
    dpe_ok = dpe_bat.dropna(subset=["lat", "lon"]).reset_index(drop=True)

    if valid.empty or dpe_ok.empty:
        return result

    dvf_rad = np.radians(valid[["lat", "lon"]].to_numpy())
    dpe_rad = np.radians(dpe_ok[["lat", "lon"]].to_numpy())

    tree = BallTree(dpe_rad, metric="haversine")
    dist, idx = tree.query(dvf_rad, k=1)

    dist_m = dist[:, 0] * EARTH_RADIUS_M
    match_idx = idx[:, 0].astype(float)
    match_idx[dist_m > max_dist_m] = np.nan

    dpe_idx = pd.Series(match_idx, index=valid.index)

    result.loc[valid.index, "dpe_median"] = dpe_idx.map(dpe_ok["dpe_median"])
    result.loc[valid.index, "annee_construction"] = dpe_idx.map(
        dpe_ok["annee_construction"]
    )

    match_rate = dpe_idx.notna().mean()
    print(f"Match DPE: {match_rate:.1%} ({dpe_idx.notna().sum()} / {len(valid)})")

    return result


df_dvf_filtered = merge_dpe_on_dvf(df_dvf_filtered, dpe_bat)
del dpe_bat

Match DPE: 63.1% (740537 / 1174403)


In [8]:
COLUMNS_TO_KEEP = [
    "datemut",
    "l_codinsee",
    "lon",
    "lat",
    "sbati",
    "nblocdep",
    "log_sterr",
    "dpe_median",
    "annee_construction",
    "home_rooms",
    "valeurfonc",
]

df_dvf_filtered_cols = df_dvf_filtered[COLUMNS_TO_KEEP]
del df_dvf_filtered

In [9]:
# dropna without requiring DPE (XGBoost handles NaN)
df_dvf_filtered_cols.dropna(
    subset=[c for c in df_dvf_filtered_cols.columns if c not in ("dpe_median", "annee_construction")],
    inplace=True,
)
df_dvf_filtered_cols.drop_duplicates(inplace=True)
df_dvf_filtered_cols.head()

,datemut,l_codinsee,lon,lat,sbati,nblocdep,log_sterr,dpe_median,annee_construction,home_rooms,valeurfonc
121669,2023-02-24,01368,4.956204,46.198676,160.0,0,9.367002,NaN,NaN,4,390000.0
121670,2023-06-23,01014,5.672540,46.270868,114.0,0,5.996452,3.0,2009.0,5,239000.0
121672,2023-08-21,01143,6.146572,46.373696,200.0,0,7.435438,NaN,NaN,5,1299850.0
121675,2023-11-15,01236,5.069958,46.341429,95.0,0,5.634790,4.0,2006.0,4,150000.0
121688,2023-02-23,01024,5.165522,46.280486,133.0,0,6.894670,4.0,1948.0,4,385000.0


In [10]:
df_dvf_filtered_cols.describe()

,lon,lat,sbati,nblocdep,log_sterr,dpe_median,annee_construction,home_rooms,valeurfonc
count,1.174402e+06,1.174402e+06,1.174402e+06,1.174402e+06,1.174402e+06,740536.000000,740536.000000,1.174402e+06,1.174402e+06
mean,1.919893e+00,4.697209e+01,1.012870e+02,5.238257e-01,6.546126e+00,4.063916,1973.633507,4.014445e+00,2.553806e+05
std,2.646358e+00,2.267439e+00,4.108520e+01,8.468804e-01,1.187895e+00,1.396873,28.819189,1.001835e+00,2.182222e+05
min,-5.131238e+00,4.136653e+01,1.100000e+01,0.000000e+00,0.000000e+00,1.000000,1850.000000,1.000000e+00,1.009000e+04
25%,-1.235436e-01,4.509704e+01,7.600000e+01,0.000000e+00,5.843544e+00,3.000000,1948.000000,3.000000e+00,1.313000e+05
50%,2.177858e+00,4.729104e+01,9.400000e+01,0.000000e+00,6.492240e+00,4.000000,1974.000000,4.000000e+00,2.064950e+05
75%,3.742969e+00,4.880167e+01,1.200000e+02,1.000000e+00,7.204893e+00,5.000000,2000.000000,5.000000e+00,3.100000e+05
max,9.556834e+00,5.108023e+01,9.900000e+02,2.000000e+02,1.081978e+01,7.000000,2021.000000,5.000000e+00,9.000000e+06


## Train

In [ ]:
features = [
    "lon", "lat", "sbati", "l_codinsee",
    "home_rooms", "nblocdep", "log_sterr",
    "dpe_median", "annee_construction",
]

required_cols = [c for c in features if c not in ("dpe_median", "annee_construction")]

df_data = (
    df_dvf_filtered_cols
    .sort_values("datemut")
    .dropna(subset=required_cols + ["valeurfonc"])
    .copy()
)

df_data["l_codinsee"] = df_data["l_codinsee"].astype("category")

print("Rows with DPE:", df_data["dpe_median"].notna().mean())

X = df_data[features]
y = np.log(df_data["valeurfonc"])

cutoff = int(len(df_data) * 0.8)
X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]

# Monotone constraints (ml/repif_ml): +1 = price ↑, -1 = price ↓
# dpe_median: A=1 … G=7 → worse class = higher value → lower price
MONOTONE = {"sbati": 1, "log_sterr": 1, "dpe_median": -1}
monotone_constraints = tuple(MONOTONE.get(f, 0) for f in features)
print("Monotone constraints:", dict(zip(features, monotone_constraints)))

param_dist = {
    "n_estimators": randint(200, 1200),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0, 5),
    "reg_lambda": uniform(0, 5),
}

search = RandomizedSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        n_jobs=-1,
        enable_categorical=True,
        monotone_constraints=monotone_constraints,
    ),
    param_distributions=param_dist,
    n_iter=20,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=1,
    verbose=2,
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV score (neg MAE log):", search.best_score_)

model_houses = search.best_estimator_

y_pred = np.exp(model_houses.predict(X_test))
y_true = np.exp(y_test)

print("R² :", r2_score(y_true, y_pred))
print("MAE:", mean_absolute_error(y_true, y_pred))
print("MAPE:", np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

Rows with DPE: 0.6305643212460469
Fitting 2 folds for each of 10 candidates, totalling 20 fits
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.15639878836228102, max_depth=7, min_child_weight=5, n_estimators=814, reg_lambda=2.229163764267956, subsample=0.6399899663272012; total time=   9.9s
[CV] END colsample_bytree=0.749816047538945, gamma=4.75357153204958, learning_rate=0.15639878836228102, max_depth=7, min_child_weight=5, n_estimators=814, reg_lambda=2.229163764267956, subsample=0.6399899663272012; total time=  17.5s
[CV] END colsample_bytree=0.7836995567863468, gamma=1.668543055695109, learning_rate=0.038573363584388155, max_depth=5, min_child_weight=6, n_estimators=508, reg_lambda=4.8495492608099715, subsample=0.9329770563201687; total time=   7.3s
[CV] END colsample_bytree=0.7836995567863468, gamma=1.668543055695109, learning_rate=0.038573363584388155, max_depth=5, min_child_weight=6, n_estimators=508, reg_lambda=4.8495492608099715, subsample=

KeyboardInterrupt: 

In [ ]:
importance_houses = pd.Series(
    model_houses.feature_importances_, index=features
).sort_values(ascending=False)
print(importance_houses)
importance_houses.plot(kind="bar", title="Feature importance — houses")